<div style="padding: 20px; background: linear-gradient(90deg, #c407bbff 0%, #3568f3ff 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🕸️ Module 7.5: Graph RAG</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Augmenting Vector Search with Knowledge Graphs for multi-hop logic.</p>
</div>

---

## 1. Where Vectors Fail

Vector databases are incredible for finding semantic similarity. But they are completely blind to **relationships**.
If you ask: *"Who are the co-authors of the authors who cited Albert Einstein?"*

A vector DB will just search for the words "Einstein" and "authors". It cannot "hop" from one node to another. 

## 2. Knowledge Graphs
Graph RAG utilizes Graph Databases (like **Neo4j**) to store data as `(Node)-[RELATION]->(Node)`. 
To demonstrate this locally without forcing you to install Neo4j server, we will simulate a graph traversal in Python, and then use the LLM to parse the final answer!

### Course alignment and free-first stack

- Covers: Graph RAG concepts and multi-hop reasoning over entity relationships.
- Runtime stack: Groq chat models for generation/evaluation when an LLM is needed, plus local Hugging Face sentence-transformers embeddings for retrieval.
- No paid OpenAI API key is required. Set `GROQ_API_KEY` only for notebooks that call an LLM; pure retrieval and embedding notebooks run locally after model weights are available.
- Current LangChain pattern: provider split packages such as `langchain_groq`, `langchain_huggingface`, and `langchain_chroma`, with runnable `.invoke()` APIs.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

# 1. Simulated Graph Database (Entities and Relationships)
KNOWLEDGE_GRAPH = {
    "Albert Einstein": {"DEVELOPED": ["Relativity"], "DEBATED_WITH": ["Niels Bohr"]},
    "Niels Bohr":      {"CONTRIBUTED_TO": ["Quantum Mechanics"], "DEBATED_WITH": ["Albert Einstein"]},
    "Werner Heisenberg": {"CONTRIBUTED_TO": ["Quantum Mechanics"]},
}

def simple_graph_query(entity: str, relation: str) -> list:
    node = KNOWLEDGE_GRAPH.get(entity, {})
    return node.get(relation, [])

print("Graph DB Loaded.")

## 3. The Multi-Hop Query
We want to know: *"Who debated with the people who developed Relativity?"*
This requires two hops:
1. `(Who) -[DEVELOPED]-> (Relativity)`
2. `(Who) -[DEBATED_WITH]-> (?)`

In [ ]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # --- HOP 1 ---
    relativity_devs = [e for e, rels in KNOWLEDGE_GRAPH.items() if "Relativity" in rels.get("DEVELOPED", [])]
    print(f"HOP 1: Developers of Relativity → {relativity_devs}")
    
    # --- HOP 2 ---
    graph_context = ""
    for dev in relativity_devs:
        debaters = simple_graph_query(dev, "DEBATED_WITH")
        graph_context += f"{dev} debated with: {', '.join(debaters)}\n"
        print(f"HOP 2: Debaters of {dev} → {debaters}")
        
    # --- PASS TO LLM ---
    print("\n--- GENERATING FINAL ANSWER ---")
    prompt = ChatPromptTemplate.from_template("""
    Answer the question based purely on the provided knowledge graph context.
    Graph Context:
    {context}
    
    Question: Who debated with the developers of Relativity?
    """)
    
    answer = (prompt | llm | StrOutputParser()).invoke({"context": graph_context})
    print(answer)
else:
    print("GROQ_API_KEY missing.")